# Cross-condition transfer

Trains YOLO11n on one defoliation state and evaluates on both, in each
direction. The gap between matched and mismatched conditions measures
robustness to a shift that occurs naturally within a single growing season.

Two runs, each on 15,000 tiles from a single field, 50 epochs. Checkpoints
write to Drive, so a recycled runtime costs at most five epochs; re-run
cells 1--4 and use the resume cell if that happens.

Needs in Drive: both train zips, both silver label zips, `raw_preds.json` per
field, `gold_v2_tiles.zip`, `gold_v2_verified.zip`, `test_classes.csv`.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
!pip install -q ultralytics
import os, json, glob, random, shutil, re
import numpy as np, pandas as pd
print('ok')

In [ ]:
# 2. Unpack tiles and silver labels
for c in ['pre','post']:
    if len(glob.glob(f'/content/imgs_{c}/train_tiles/*.jpg')) == 0:
        !rm -rf /content/imgs_{c}
        !unzip -q {DRIVE}/cotton_train_{c}.zip -d /content/imgs_{c}
    if len(glob.glob(f'/content/lbls_{c}/silver_{c}_labels/*.txt')) == 0:
        !rm -rf /content/lbls_{c}
        !unzip -q {DRIVE}/silver_{c}/labels.zip -d /content/lbls_{c}
for c in ['pre','post']:
    print(c, len(glob.glob(f'/content/imgs_{c}/train_tiles/*.jpg')), 'images')

In [ ]:
# 3. Per-field training subsets, 15,000 tiles each, stratified by density
def dbin(n):
    return ('0' if n==0 else '1-50' if n<=50 else '51-150' if n<=150
            else '151-300' if n<=300 else '300+')

subsets = {}
for c in ['pre','post']:
    preds = json.load(open(f'{DRIVE}/silver_{c}/raw_preds.json'))
    strata = {}
    for t, boxes in preds.items():
        strata.setdefault(dbin(len(boxes)), []).append(t)
    tot = sum(len(v) for v in strata.values())
    random.seed(202)
    sel = []
    for b, lst in sorted(strata.items()):
        k = min(len(lst), max(50, round(15000*len(lst)/tot)))
        sel.extend(random.sample(lst, k))
    random.shuffle(sel)
    subsets[c] = dict(val=sel[:750], train=sel[750:])
    print(c, 'train', len(subsets[c]['train']), 'val', len(subsets[c]['val']))
json.dump(subsets, open(f'{DRIVE}/transfer_subsets.json','w'))

In [ ]:
# 4. Assemble one dataset per field
for c in ['pre','post']:
    root = f'/content/ds_{c}'
    !rm -rf {root}
    for s in ['train','val']:
        for sub in ['images','labels']:
            os.makedirs(f'{root}/{s}/{sub}', exist_ok=True)
    miss = 0
    for s, tiles in subsets[c].items():
        for t in tiles:
            si = f'/content/imgs_{c}/train_tiles/{t}'
            sl = f'/content/lbls_{c}/silver_{c}_labels/{os.path.splitext(t)[0]}.txt'
            if not (os.path.exists(si) and os.path.exists(sl)):
                miss += 1; continue
            os.symlink(si, f'{root}/{s}/images/{t}')
            shutil.copy(sl, f'{root}/{s}/labels/')
    with open(f'{root}/data.yaml','w') as f:
        f.write(f'path: {root}\ntrain: train/images\nval: val/images\n'
                'names:\n  0: cotton_boll\n')
    print(c, len(os.listdir(f'{root}/train/images')),
          len(os.listdir(f'{root}/val/images')), 'missing', miss)

## 5. Train

Run once with `FIELD = 'pre'`, then again with `FIELD = 'post'`.

In [ ]:
FIELD = 'post'          # <-- change to 'post' for the second run
RUN = f'yolo11n_{FIELD}only'
from ultralytics import YOLO

m = YOLO('yolo11n.pt')
m.train(data=f'/content/ds_{FIELD}/data.yaml', epochs=50, imgsz=640, batch=16,
        patience=20, project=f'{DRIVE}/runs', name=RUN,
        max_det=1000, seed=42, val=True, plots=True, save_period=5,
        exist_ok=True)

dst = f'{DRIVE}/baselines/{RUN}'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'{DRIVE}/runs/{RUN}/weights/best.pt', f'{dst}/best.pt')
for f in ['results.csv','args.yaml']:
    p = f'{DRIVE}/runs/{RUN}/{f}'
    if os.path.exists(p): shutil.copy(p, dst)
print('saved to', dst)

In [ ]:
# Resume cell, if the runtime was recycled mid-training
# from ultralytics import YOLO
# FIELD = 'pre'
# YOLO(f'{DRIVE}/runs/yolo11n_{FIELD}only/weights/last.pt').train(resume=True)

## 6. Evaluate both models on both halves of the verified set

Matched condition is train and test on the same defoliation state;
mismatched is the cross-field direction. Scored on the hand-verified gold
tiles, so the comparison rests on human labels rather than model output.

In [ ]:
from tqdm import tqdm
from collections import defaultdict

!rm -rf /content/gold /content/gold_ann
!unzip -q {DRIVE}/gold_v2_tiles.zip -d /content/gold
!unzip -q {DRIVE}/gold_v2_verified.zip -d /content/gold_ann
TILES = '/content/gold/gold_v2'
ann = glob.glob('/content/gold_ann/**/instances_default.json', recursive=True)[0]
verified = json.load(open(ann))

names = {im['id']: im['file_name'].split('/')[-1] for im in verified['images']}
gt = defaultdict(list)
for a in verified['annotations']:
    x, y, w, h = a['bbox']
    gt[names[a['image_id']]].append([x, y, x+w, y+h])
for n in names.values(): gt.setdefault(n, [])

cls_of_stem = {}
import csv
with open(f'{DRIVE}/test_classes.csv') as f:
    for row in csv.DictReader(f):
        cls_of_stem[row['stem']] = row['class']
def stem_of(n): return re.split(r'_r\d+_c\d+_x\d+_y\d+', n)[0]
field_of = {t: cls_of_stem.get(stem_of(t), '?') for t in gt}
groups = {'pre':  [t for t in gt if field_of[t]=='predefoliation'  and gt[t]],
          'post': [t for t in gt if field_of[t]=='postdefoliation' and gt[t]]}
print({k: len(v) for k, v in groups.items()})

In [ ]:
from ultralytics import YOLO
def iou_matrix(A, B):
    if len(A)==0 or len(B)==0: return np.zeros((len(A), len(B)), np.float32)
    A = np.asarray(A, np.float32); B = np.asarray(B, np.float32)
    ix1 = np.maximum(A[:,0,None], B[None,:,0]); iy1 = np.maximum(A[:,1,None], B[None,:,1])
    ix2 = np.minimum(A[:,2,None], B[None,:,2]); iy2 = np.minimum(A[:,3,None], B[None,:,3])
    inter = np.clip(ix2-ix1,0,None)*np.clip(iy2-iy1,0,None)
    aA = ((A[:,2]-A[:,0])*(A[:,3]-A[:,1]))[:,None]
    aB = ((B[:,2]-B[:,0])*(B[:,3]-B[:,1]))[None,:]
    u = aA + aB - inter
    return np.divide(inter, u, out=np.zeros_like(inter), where=u>0)

def prf(preds, gtd, thr, keys, iou_t=0.5):
    TP=FP=FN=0
    for t in keys:
        g = gtd.get(t, [])
        p = sorted([b for b in preds.get(t, []) if b[4] >= thr], key=lambda b:-b[4])
        if not p: FN += len(g); continue
        if not g: FP += len(p); continue
        M = iou_matrix([b[:4] for b in p], g)
        taken=set(); tp=0
        for i in range(len(p)):
            for cand in np.argsort(-M[i]):
                if M[i,cand] < iou_t: break
                if cand not in taken:
                    taken.add(int(cand)); tp+=1; break
        TP+=tp; FP+=len(p)-tp; FN+=len(g)-tp
    P=TP/max(TP+FP,1); R=TP/max(TP+FN,1)
    return dict(P=round(P,4), R=round(R,4), F1=round(2*P*R/max(P+R,1e-9),4))

def predict(weights, tiles_list):
    mdl = YOLO(weights); out={}
    for i in tqdm(range(0, len(tiles_list), 64)):
        chunk = tiles_list[i:i+64]
        res = mdl.predict([f'{TILES}/{t}' for t in chunk], imgsz=640, conf=0.001,
                          iou=0.7, max_det=1000, verbose=False)
        for t, r in zip(chunk, res):
            b = r.boxes
            out[t] = [[float(x1),float(y1),float(x2),float(y2),float(c)]
                      for (x1,y1,x2,y2), c in zip(b.xyxy.cpu().numpy(),
                                                  b.conf.cpu().numpy())]
    return out

all_gold = groups['pre'] + groups['post']
rows = []
for trained_on in ['pre','post']:
    w = f'{DRIVE}/baselines/yolo11n_{trained_on}only/best.pt'
    if not os.path.exists(w):
        print('missing', w); continue
    pr = predict(w, all_gold)
    for tested_on in ['pre','post']:
        m = prf(pr, gt, 0.30, groups[tested_on])
        rows.append(dict(trained_on=trained_on, tested_on=tested_on,
                         matched=(trained_on==tested_on),
                         n_tiles=len(groups[tested_on]), **m))

# both-field reference
w_both = f'{DRIVE}/baselines/yolo11n/best.pt'
pr_both = predict(w_both, all_gold)
for tested_on in ['pre','post']:
    m = prf(pr_both, gt, 0.30, groups[tested_on])
    rows.append(dict(trained_on='both', tested_on=tested_on, matched=None,
                     n_tiles=len(groups[tested_on]), **m))

res = pd.DataFrame(rows)
print(res.to_string(index=False))
res.to_csv(f'{DRIVE}/results_transfer.csv', index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
!pip install -q ultralytics
import os, json, glob, random, shutil, re
import numpy as np, pandas as pd
print('ok')

The difference between matched and mismatched rows is the transfer gap.
The `both` rows show what training on the combined corpus recovers, and
bound how much of any gap is attributable to condition shift rather than to
the smaller per-field training sets.